In [1]:
import os
import pandas as pd

from os       import makedirs
from tqdm     import tqdm
from datasets import load_dataset

# prepare data
dataset = load_dataset("arch-be/brabant-xvii", name="doc_by_doc")
train   = dataset['train'].to_pandas()
valid   = dataset['valid'].to_pandas()
test    = dataset['test'].to_pandas()
dataset = pd.concat([train, test, valid]).set_index(['project', 'file_id'])
# start with the longest texts (fail fast)
dataset.sort_values(by='text', key=lambda x: x.str.len(), ascending=False, inplace=True)

with open('prompt.txt', encoding='utf8') as f:
    prompt = f.read()


/home/xgillard/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from groq import Groq

client = Groq(api_key="USE_YOUR_OWN_KEY")

In [3]:
model = "deepseek-r1-distill-llama-70b"

makedirs(f'generated_texts/{model}/', exist_ok=True)
#
for id, text in tqdm(dataset['text'].items()):
    project,file_id = id
    fname = f'generated_texts/{model}/{project}_{file_id}'
    if os.path.exists(fname):
        continue
    else:
        with open(fname, 'w', encoding='utf8') as f:
            messages = [
                    {'role': 'user',   'content': f"{prompt}\n## Document\n{text}"},
                ]
            response = client.chat.completions.create(
                model=model,
                response_format={'type': 'json_object'},
                messages = messages,
            )
            print(response, file=f)

0it [00:00, ?it/s]


NotFoundError: Error code: 404 - {'error': {'message': 'The model `deepseek-r1-distill-llama-70b-specdec` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}